In [1]:
# setup for the GIF notebook (self-contained)
# Loads the same model + arrays the analysis notebook uses, so the GIF cells
# (which expect model, latents_d8_k9, centroids_d8_k9, labels_d8_k9,
#  reducer_3d, cluster_colors) run standalone.


# imports (everything the GIF cells use)
import os
import numpy as np
import torch
import cv2
import joblib
import matplotlib
matplotlib.use("Agg")                       # headless backend for frame rendering
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.mplot3d import proj3d      # used by screen_xy() for label projection
from PIL import Image
import imageio.v2 as imageio
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"

os.chdir(ROOT)

sys.path.insert(0, str(SRC_DIR))

from vae_model import ResidualVAE

# paths (identical to the analysis notebook)
vae_main_path  = str(ARTIFACT_DIR)
model_weights  = os.path.join(vae_main_path, "best_ResidualVAE_d8.pth")
centroids_path = os.path.join(vae_main_path, "centroids_8d_K9.npy")
labels_path    = os.path.join(vae_main_path, "cluster_labels_K9.npy")
latents_path   = os.path.join(vae_main_path, "latents_8d.npy")
umap_path      = os.path.join(vae_main_path, "umap_reducer_3d.joblib")

# model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = ResidualVAE()
state_dict = torch.load(model_weights, map_location=DEVICE, weights_only=False)
model.load_state_dict(state_dict['model_state'])
model.to(DEVICE)
model.eval()

# arrays (same names + dtypes the GIF cells expect)
latents_d8_k9   = torch.tensor(np.load(latents_path),   dtype=torch.float32)
centroids_d8_k9 = torch.tensor(np.load(centroids_path), dtype=torch.float32)
labels_d8_k9    = torch.tensor(np.load(labels_path),    dtype=torch.float32)
reducer_3d      = joblib.load(umap_path)

# cluster palette (same as analysis)
cluster_colors = {0:'#d4354f', 1:'#61b058', 2:'#fbe050', 3:'#4866d1', 4:'#e78745',
                  5:'#6ed2f0', 6:'#dd4cdf', 7:'#c9ec63', 8:'#888888'}

print("setup OK:",
      "latents", tuple(latents_d8_k9.shape),
      "| centroids", tuple(centroids_d8_k9.shape),
      "| labels", tuple(labels_d8_k9.shape),
      "| device", DEVICE)

setup OK: latents (22400, 8) | centroids (9, 8) | labels (22400,) | device cpu


In [2]:
import os
SNAP_DIR = "./data/baseline/"      # the current guess
print("exists:", os.path.isdir(SNAP_DIR))
if os.path.isdir(SNAP_DIR):
    files = sorted(os.listdir(SNAP_DIR))
    print("n files:", len(files), "| first:", files[:3])

exists: True
n files: 3 | first: ['.DS_Store', 'rot_sensor_data.txt', 'snapshots']


In [4]:
# Baseline latent-space evolution  ->  GIF/MP4
#   BIG 3-D UMAP skeleton with the trajectory riding through it; vorticity frame
#   as a small top-right tag. Time label in SHEDDING CYCLES, starting at 0.
#
# Run after the setup cell (model, latents_d8_k9, centroids_d8_k9,
# labels_d8_k9, reducer_3d, cluster_colors defined).

from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.patheffects as pe
import imageio.v2 as imageio

# PATHS / IO
SNAP_DIR  = "./data/baseline/snapshots/"
SNAP_TMPL = "snapshot_{it}.png"                 # unpadded names (snapshot_80000.png)
IT_START, IT_END, IT_STEP = 80000, 100000, 50
SAVE_DIR  = "./gifs"                            # <<< output folder
os.makedirs(SAVE_DIR, exist_ok=True)
GIF_PATH  = os.path.join(SAVE_DIR, "baseline_latent_evolution.gif")
MP4_PATH  = os.path.join(SAVE_DIR, "baseline_latent_evolution.mp4")

# KNOBS (match slide-3 geometry)
MARGIN_THRESHOLD = 2.0
VIEW_ELEV, VIEW_AZIM = 0, 50
ZOOM = 0.88
SOLID_MAX, DASH_MAX = 1.75, 1.85
DT = 0.001
T_SH = 1.667                                    # shedding period (t.u.)
ITER_PER_CYCLE = T_SH / DT                      # iters per shedding cycle (~1667)
REGIME_LABEL = "baseline"                       # shown in the time/state label

# pacing
FRAME_STRIDE = 2
FPS = 11
TRAIL_LEN = 18
HOLD_SECONDS  = 3.0
FINAL_PT_SIZE = 20

# layout
FIGSIZE = (7.6, 7.0)
AX_RECT  = [-0.06, -0.06, 1.12, 1.12]
TAG_RECT = [0.655, 0.775, 0.335, 0.185]

# styling
BG_ALPHA, BG_SIZE = 0.14, 5
BRIGHT_ALPHA, BRIGHT_SIZE = 0.95, 9
CEN_SIZE = 46
HEAD_SIZE = 95
EDGE_COLOR, EDGE_ALPHA = "0.72", 0.9
PATH_COLOR, PATH_ALPHA = "0.55", 0.5

# encoder crop + gray-removal thresholds
CROP_H, CROP_W = (87, 215), (46, 366)
AIRFOIL_LUMA_THRESH, COLORED_SAT_THRESH = 80, 0.14
GRAY_SAT_THRESH, GRAY_LUMA_THRESH = 0.16, 110

cluster_colors = {0:'#d4354f',1:'#61b058',2:'#fbe050',3:'#4866d1',4:'#e78745',
                  5:'#6ed2f0',6:'#dd4cdf',7:'#c9ec63',8:'#888888'}

# NOTEBOOK ARRAYS
latents_np   = latents_d8_k9.numpy().astype(np.float32)
centroids_np = centroids_d8_k9.numpy().astype(np.float32)
labels_np    = labels_d8_k9.numpy().astype(int)
DEVICE = next(model.parameters()).device

emb_pts = reducer_3d.transform(latents_np)
emb_cen = reducer_3d.transform(centroids_np)
Dc = np.linalg.norm(centroids_np[:,None,:] - centroids_np[None,:,:], axis=2)

D   = np.linalg.norm(latents_np[:,None,:] - centroids_np[None,:,:], axis=2)
own = D[np.arange(len(labels_np)), labels_np]
D2  = D.copy(); D2[np.arange(len(labels_np)), labels_np] = np.inf
margin_ratio = D2.min(axis=1) / own
keep = np.ones(len(labels_np), bool) if MARGIN_THRESHOLD is None else (margin_ratio >= MARGIN_THRESHOLD)

# HELPERS
def load_crop(png_path):
    img = Image.open(png_path)
    if img.mode != "RGB": img = img.convert("RGB")
    a = np.array(img)
    return a[CROP_H[0]:CROP_H[1], CROP_W[0]:CROP_W[1], :]

def encode(cropped):
    model.eval()
    with torch.no_grad():
        x = torch.as_tensor(cropped, dtype=torch.float32, device=DEVICE) / 255.0
        x = x.permute(2,0,1).unsqueeze(0)
        return model.encode(x).squeeze(0).cpu().numpy()

def clean_field(cropped):
    rgb = cropped.astype(np.float32)
    R,G,B = rgb[...,0], rgb[...,1], rgb[...,2]
    mx = rgb.max(2); mn = rgb.min(2); chroma = mx-mn
    luma = 0.299*R + 0.587*G + 0.114*B
    sat  = chroma/(mx+1e-6)
    airfoil = luma < AIRFOIL_LUMA_THRESH
    colored = sat > COLORED_SAT_THRESH
    gray_halo = (sat < GRAY_SAT_THRESH) & (luma > GRAY_LUMA_THRESH)
    mask = (colored | airfoil); mask[gray_halo] = False
    mask = mask.astype(np.uint8)
    k = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    alpha = np.clip(cv2.GaussianBlur(mask.astype(np.float32), (0,0), 0.7), 0, 1)
    rgb_clean = rgb.copy(); rgb_clean[gray_halo] = 255
    out = rgb_clean*alpha[...,None] + 255*(1-alpha[...,None])
    return np.clip(out, 0, 255).astype(np.uint8)

def screen_xy(axis, p3):
    x,y,_ = proj3d.proj_transform(p3[0],p3[1],p3[2], axis.get_proj())
    return np.array([x,y])

# BUILD TRAJECTORY
its = list(range(IT_START, IT_END+1, IT_STEP))[::FRAME_STRIDE]
paths = [os.path.join(SNAP_DIR, SNAP_TMPL.format(it=it)) for it in its]
paths = [p for p in paths if os.path.exists(p)]
assert len(paths) > 0, f"No snapshots in {SNAP_DIR} matching {SNAP_TMPL}"
print(f"[traj] {len(paths)} snapshots")

# iteration of each used frame -> elapsed shedding cycles from 0
it_used   = [int(os.path.basename(p).split("_")[1].split(".")[0]) for p in paths]
it0       = it_used[0]
cyc_used  = [(it - it0) / ITER_PER_CYCLE for it in it_used]   # cycles from 0
CYC_END   = (it_used[-1] - it0) / ITER_PER_CYCLE

traj_z, disp_imgs = [], []
for p in paths:
    crop = load_crop(p)
    traj_z.append(encode(crop))
    disp_imgs.append(clean_field(crop))
traj_z = np.asarray(traj_z, np.float32)
traj_emb = reducer_3d.transform(traj_z)
traj_clu = np.argmin(np.linalg.norm(traj_z[:,None,:] - centroids_np[None,:,:], axis=2), axis=1)

# STATIC BACKGROUND (once)
fig = plt.figure(figsize=FIGSIZE, dpi=110, facecolor="white")
ax  = fig.add_axes([0,0,1,1], projection="3d", facecolor="white"); ax.set_axis_off()
halo = [pe.withStroke(linewidth=2.5, foreground="white")]

for k in range(9):
    m = keep & (labels_np == k)
    ax.scatter(emb_pts[m,0], emb_pts[m,1], emb_pts[m,2],
               s=BG_SIZE, c=cluster_colors[k], alpha=BG_ALPHA,
               edgecolors="none", depthshade=False)
ax.scatter(emb_cen[:,0], emb_cen[:,1], emb_cen[:,2], s=CEN_SIZE,
           c=[cluster_colors[k] for k in range(9)], marker="D",
           edgecolors="black", linewidths=1.0, depthshade=False, zorder=5)

ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
try:
    ax.set_box_aspect((np.ptp(emb_pts[:,0]), np.ptp(emb_pts[:,1]), np.ptp(emb_pts[:,2])))
except Exception:
    pass
for setlim, lo, hi in [
    (ax.set_xlim3d, emb_pts[:,0].min(), emb_pts[:,0].max()),
    (ax.set_ylim3d, emb_pts[:,1].min(), emb_pts[:,1].max()),
    (ax.set_zlim3d, emb_pts[:,2].min(), emb_pts[:,2].max())]:
    c = 0.5*(lo+hi); r = 0.5*(hi-lo)*ZOOM; setlim(c-r, c+r)

EDGES = [(5,2),(2,8),(8,1),(1,3),(3,5),(2,3),(5,8),
         (0,1),(0,2),(0,3),(0,5),(0,8),(0,4),(6,4),(6,7)]
EDGES = [(i,j) for (i,j) in EDGES if Dc[i,j] <= DASH_MAX]
_w = np.array([Dc[i,j] for i,j in EDGES]); wmin, wmax = _w.min(), _w.max()
def lw_for(d):
    t = 0.0 if wmax == wmin else (d-wmin)/(wmax-wmin)
    return 2.4 + t*(0.6-2.4)
for i,j in EDGES:
    style = "-" if Dc[i,j] < SOLID_MAX else "--"
    xi, xj = emb_cen[i], emb_cen[j]
    ax.plot([xi[0],xj[0]],[xi[1],xj[1]],[xi[2],xj[2]],
            color=EDGE_COLOR, lw=lw_for(Dc[i,j]), linestyle=style,
            alpha=EDGE_ALPHA, zorder=3)

ax.set_position(AX_RECT)
fig.canvas.draw()
for k in range(9):
    s = screen_xy(ax, emb_cen[k])
    ax.annotate(f"C{k}", xy=(s[0],s[1]), xytext=(0,9), textcoords="offset points",
                color=cluster_colors[k], fontsize=11, fontweight="bold",
                zorder=30, annotation_clip=False, path_effects=halo)

axim = fig.add_axes(TAG_RECT); axim.set_xticks([]); axim.set_yticks([])
for sp in axim.spines.values(): sp.set_edgecolor("0.7"); sp.set_linewidth(1.0)
im_handle = axim.imshow(disp_imgs[0], aspect="auto")

txt = fig.text(0.03, 0.95, "", fontsize=12, fontweight="bold", family="monospace")

# ANIMATE
frames = []
dyn = []
for f in range(len(traj_emb)):
    for a in dyn: a.remove()
    dyn = []

    c = int(traj_clu[f])
    mb = keep & (labels_np == c)
    ov = ax.scatter(emb_pts[mb,0], emb_pts[mb,1], emb_pts[mb,2],
                    s=BRIGHT_SIZE, c=cluster_colors[c], alpha=BRIGHT_ALPHA,
                    edgecolors="none", depthshade=False, zorder=6)
    dyn.append(ov)

    if f >= 1:
        vp = traj_emb[:f+1]
        (ln,) = ax.plot(vp[:,0], vp[:,1], vp[:,2], color=PATH_COLOR,
                        lw=1.0, alpha=PATH_ALPHA, zorder=7); dyn.append(ln)
    lo = max(0, f - TRAIL_LEN); tr = traj_emb[lo:f+1]
    if len(tr) >= 2:
        (tl,) = ax.plot(tr[:,0], tr[:,1], tr[:,2], color=cluster_colors[c],
                        lw=2.6, alpha=0.95, zorder=8); dyn.append(tl)
    hz = traj_emb[f]
    hd = ax.scatter([hz[0]],[hz[1]],[hz[2]], s=HEAD_SIZE, c=cluster_colors[c],
                    marker="o", edgecolors="black", linewidths=1.4,
                    depthshade=False, zorder=9); dyn.append(hd)

    im_handle.set_data(disp_imgs[f])
    txt.set_text(f"{REGIME_LABEL}   t = {cyc_used[f]:5.2f} cycles   C{c}")

    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[...,:3].copy())

# FINAL held frame
for a in dyn: a.remove()
(lnf,) = ax.plot(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2],
                 color=PATH_COLOR, lw=1.0, alpha=PATH_ALPHA, zorder=7)
cols = [cluster_colors[int(c)] for c in traj_clu]
scf = ax.scatter(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2],
                 s=FINAL_PT_SIZE, c=cols, alpha=0.95,
                 edgecolors="none", depthshade=False, zorder=8)
im_handle.set_data(disp_imgs[-1])
txt.set_text(f"{REGIME_LABEL}   full trajectory  (t = 0.00 - {CYC_END:.2f} cycles)")
fig.canvas.draw()
final_frame = np.asarray(fig.canvas.buffer_rgba())[...,:3].copy()
frames.extend([final_frame] * max(1, int(HOLD_SECONDS * FPS)))

# SAVE
imageio.mimsave(GIF_PATH, frames, fps=FPS, loop=1)
print("wrote", GIF_PATH, f"({len(frames)} frames)")
try:
    imageio.mimsave(MP4_PATH, frames, fps=FPS, codec="libx264",
                    quality=8, macro_block_size=16)
    print("wrote", MP4_PATH)
except Exception as e:
    print("mp4 skipped -- install the ffmpeg backend:  pip install imageio[ffmpeg]")
    print("  (details:", e, ")")
plt.close(fig)

[traj] 200 snapshots


/opt/anaconda3/envs/torch_env/lib/python3.10/site-packages/torch/nn/modules/conv.py:455: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1742433317015/work/aten/src/ATen/native/Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,


wrote ./gifs/baseline_latent_evolution.gif (233 frames)
mp4 skipped -- install the ffmpeg backend:  pip install imageio[ffmpeg]
  (details: Could not find a backend to open `./gifs/baseline_latent_evolution.mp4`` with iomode `wI`.
Based on the extension, the following plugins might add capable backends:
  FFMPEG:  pip install imageio[ffmpeg]
  pyav:  pip install imageio[pyav] )


In [5]:
# Latent-space evolution GIF/MP4 -- baseline / hold / escape (one file).
# Time in SHEDDING CYCLES (from 0). Matched to 9_K_exploration.ipynb.
#
# Run INSIDE the notebook (needs: model, centroids_d8_k9, latents_d8_k9,
# labels_d8_k9, reducer_3d, cluster_colors, functions.load_file).
# import os
# import numpy as np
# import torch, cv2
# import matplotlib; matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# import matplotlib.patheffects as pe
# from mpl_toolkits.mplot3d import proj3d              # noqa
# import imageio.v2 as imageio
from functions import load_file                      # SAME crop as your encoder

# PICK THE REGIME
REGIME = "hold"          # "baseline" | "hold" | "escape"

# all runs are local to this folder (no absolute desktop paths)
CFG = {
  "baseline": dict(
      run_dir=None,
      snap_dir="./data/baseline",
      it=(80_000, 100_000, 50), title="baseline"),
  "hold": dict(
      run_dir="./data/top_c0_runs/classic-sweep-4",
      snap_dir="./data/top_c0_runs/classic-sweep-4/snapshots",
      it=None, title="hold  (lock-on)"),
  "escape": dict(
      run_dir="./data/top_c6_runs/worthy-sweep-43",
      snap_dir="./data/top_c6_runs/worthy-sweep-43/snapshots",
      it=None, title="escape  (relaxation oscillator)"),
}
cfg = CFG[REGIME]
SNAP_DIR, TITLE = cfg["snap_dir"], cfg["title"]
RUN_DIR = cfg["run_dir"]
SHOW_LAW = RUN_DIR is not None
FORCING_TXT = os.path.join(RUN_DIR, "rbf_forcing.txt") if SHOW_LAW else None
SNAP_TMPL = "snapshot_{it:06d}.png"

SAVE_DIR = "./gifs"                                   # <<< output folder
os.makedirs(SAVE_DIR, exist_ok=True)
GIF_PATH = os.path.join(SAVE_DIR, f"{REGIME}_latent_evolution.gif")
MP4_PATH = os.path.join(SAVE_DIR, f"{REGIME}_latent_evolution.mp4")

# KNOBS (match slide-3 geometry)
MARGIN_THRESHOLD = 2.0
VIEW_ELEV, VIEW_AZIM = 0, 50
ZOOM = 0.88
SOLID_MAX, DASH_MAX = 1.75, 1.85
DT = 0.001
T_SH = 1.667                                          # shedding period (t.u.)
ITER_PER_CYCLE = T_SH / DT                            # iters per cycle (~1667)

FRAME_STRIDE = 2
FPS = 11
TRAIL_LEN = 18
HOLD_SECONDS, FINAL_PT_SIZE = 3.0, 20

FIGSIZE = (10.6, 7.0) if SHOW_LAW else (7.6, 7.0)
AX_RECT  = [-0.03, -0.05, 0.72, 1.10] if SHOW_LAW else [-0.06, -0.06, 1.12, 1.12]
TAG_RECT = [0.71, 0.70, 0.27, 0.16]  if SHOW_LAW else [0.655, 0.775, 0.335, 0.185]
LAW_RECT = [0.735, 0.12, 0.245, 0.34]
LABEL_XY = (0.71, 0.90) if SHOW_LAW else (0.03, 0.95)

BG_ALPHA, BG_SIZE = 0.14, 5
BRIGHT_ALPHA, BRIGHT_SIZE = 0.95, 9
CEN_SIZE, HEAD_SIZE = 46, 95
EDGE_COLOR, EDGE_ALPHA = "0.72", 0.9
PATH_COLOR, PATH_ALPHA = "0.55", 0.5
LAW_DIM_ALPHA, LAW_LINE_COLOR = 0.15, "0.8"

cluster_colors = {0:'#d4354f',1:'#61b058',2:'#fbe050',3:'#4866d1',4:'#e78745',
                  5:'#6ed2f0',6:'#dd4cdf',7:'#c9ec63',8:'#888888'}

# NOTEBOOK ARRAYS
latents_np   = latents_d8_k9.numpy().astype(np.float32)
centroids_np = centroids_d8_k9.numpy().astype(np.float32)
labels_np    = labels_d8_k9.numpy().astype(int)
DEVICE = next(model.parameters()).device

emb_pts = reducer_3d.transform(latents_np)
emb_cen = reducer_3d.transform(centroids_np)
Dc = np.linalg.norm(centroids_np[:,None,:] - centroids_np[None,:,:], axis=2)

D   = np.linalg.norm(latents_np[:,None,:] - centroids_np[None,:,:], axis=2)
own = D[np.arange(len(labels_np)), labels_np]
D2  = D.copy(); D2[np.arange(len(labels_np)), labels_np] = np.inf
keep = np.ones(len(labels_np), bool) if MARGIN_THRESHOLD is None else ((D2.min(1)/own) >= MARGIN_THRESHOLD)

# HELPERS
def encode_and_field(png_path):
    arr = load_file(png_path)
    x = (torch.tensor(arr, dtype=torch.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        z = model.encode(x).squeeze(0).cpu().numpy()
    return z, np.asarray(arr, dtype=np.uint8)

def clean_field(rgb_u8):
    rgb = rgb_u8.astype(np.float32)
    R,G,B = rgb[...,0], rgb[...,1], rgb[...,2]
    mx = rgb.max(2); mn = rgb.min(2); chroma = mx-mn
    luma = 0.299*R + 0.587*G + 0.114*B
    sat = chroma/(mx+1e-6)
    airfoil = luma < 80; colored = sat > 0.14
    gray_halo = (sat < 0.16) & (luma > 110)
    mask = (colored | airfoil); mask[gray_halo] = False
    mask = mask.astype(np.uint8); k = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    alpha = np.clip(cv2.GaussianBlur(mask.astype(np.float32), (0,0), 0.7), 0, 1)
    rgb_clean = rgb.copy(); rgb_clean[gray_halo] = 255
    return np.clip(rgb_clean*alpha[...,None] + 255*(1-alpha[...,None]), 0, 255).astype(np.uint8)

def screen_xy(axis, p3):
    x,y,_ = proj3d.proj_transform(p3[0],p3[1],p3[2], axis.get_proj()); return np.array([x,y])

def read_forcing_txt(path):
    it, up = [], []
    with open(path) as f:
        next(f)
        for line in f:
            p = line.strip().split(",")
            if len(p) >= 2:
                it.append(int(p[0])); up.append(float(p[1]))
    return np.array(it, int), np.array(up, float)

# BUILD THE TIMELINE
if SHOW_LAW:
    its_full, amp_full = read_forcing_txt(FORCING_TXT)
else:
    a,b,s = cfg["it"]; its_full = np.arange(a, b, s); amp_full = None

sel = np.arange(len(its_full))[::FRAME_STRIDE]
its = its_full[sel]
amp = amp_full[sel] if SHOW_LAW else None

paths, keep_i = [], []
for n, it in enumerate(its):
    p = os.path.join(SNAP_DIR, SNAP_TMPL.format(it=int(it)))
    if os.path.exists(p): paths.append(p); keep_i.append(n)
keep_i = np.array(keep_i, int)
its = its[keep_i]; amp = amp[keep_i] if SHOW_LAW else None
assert len(paths) > 0, f"No snapshots in {SNAP_DIR} matching {SNAP_TMPL}"
print(f"[{REGIME}] {len(paths)} frames  (forcing rows: {len(its_full) if SHOW_LAW else 'n/a'})")

traj_z, disp_imgs = [], []
for p in paths:
    z, arr = encode_and_field(p)
    traj_z.append(z); disp_imgs.append(clean_field(arr))
traj_z = np.asarray(traj_z, np.float32)
traj_emb = reducer_3d.transform(traj_z)
traj_clu = np.argmin(np.sum((traj_z[:,None,:] - centroids_np[None,:,:])**2, axis=2), axis=1)

# time in SHEDDING CYCLES from 0
it0   = int(its[0])
cyc   = (its.astype(float) - it0) / ITER_PER_CYCLE     # <<< cycles, starts at 0
CYC_END = cyc[-1]

# STATIC BACKGROUND
fig = plt.figure(figsize=FIGSIZE, dpi=110, facecolor="white")
ax  = fig.add_axes([0,0,1,1], projection="3d", facecolor="white"); ax.set_axis_off()
halo = [pe.withStroke(linewidth=2.5, foreground="white")]

for k in range(9):
    m = keep & (labels_np == k)
    ax.scatter(emb_pts[m,0], emb_pts[m,1], emb_pts[m,2], s=BG_SIZE,
               c=cluster_colors[k], alpha=BG_ALPHA, edgecolors="none", depthshade=False)
ax.scatter(emb_cen[:,0], emb_cen[:,1], emb_cen[:,2], s=CEN_SIZE,
           c=[cluster_colors[k] for k in range(9)], marker="D",
           edgecolors="black", linewidths=1.0, depthshade=False, zorder=5)
ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
try: ax.set_box_aspect((np.ptp(emb_pts[:,0]), np.ptp(emb_pts[:,1]), np.ptp(emb_pts[:,2])))
except Exception: pass
for setlim, lo, hi in [
    (ax.set_xlim3d, emb_pts[:,0].min(), emb_pts[:,0].max()),
    (ax.set_ylim3d, emb_pts[:,1].min(), emb_pts[:,1].max()),
    (ax.set_zlim3d, emb_pts[:,2].min(), emb_pts[:,2].max())]:
    c = 0.5*(lo+hi); r = 0.5*(hi-lo)*ZOOM; setlim(c-r, c+r)

EDGES = [(5,2),(2,8),(8,1),(1,3),(3,5),(2,3),(5,8),
         (0,1),(0,2),(0,3),(0,5),(0,8),(0,4),(6,4),(6,7)]
EDGES = [(i,j) for (i,j) in EDGES if Dc[i,j] <= DASH_MAX]
_w = np.array([Dc[i,j] for i,j in EDGES]); wmin, wmax = _w.min(), _w.max()
def lw_for(d):
    t = 0.0 if wmax==wmin else (d-wmin)/(wmax-wmin); return 2.4 + t*(0.6-2.4)
for i,j in EDGES:
    style = "-" if Dc[i,j] < SOLID_MAX else "--"; xi, xj = emb_cen[i], emb_cen[j]
    ax.plot([xi[0],xj[0]],[xi[1],xj[1]],[xi[2],xj[2]], color=EDGE_COLOR,
            lw=lw_for(Dc[i,j]), linestyle=style, alpha=EDGE_ALPHA, zorder=3)

ax.set_position(AX_RECT); fig.canvas.draw()
for k in range(9):
    s = screen_xy(ax, emb_cen[k])
    ax.annotate(f"C{k}", xy=(s[0],s[1]), xytext=(0,9), textcoords="offset points",
                color=cluster_colors[k], fontsize=11, fontweight="bold",
                zorder=30, annotation_clip=False, path_effects=halo)

axim = fig.add_axes(TAG_RECT); axim.set_xticks([]); axim.set_yticks([])
for sp in axim.spines.values(): sp.set_edgecolor("0.7"); sp.set_linewidth(1.0)
im_handle = axim.imshow(disp_imgs[0], aspect="auto")

if SHOW_LAW:
    axL = fig.add_axes(LAW_RECT)
    axL.plot(cyc, amp, color=LAW_LINE_COLOR, lw=0.9, alpha=0.7, zorder=1)
    axL.scatter(cyc, amp, c=[cluster_colors[int(c)] for c in traj_clu],
                s=16, alpha=LAW_DIM_ALPHA, edgecolors="none", zorder=2)
    axL.set_xlabel("time  (shedding cycles)", fontsize=8)      # <<< cycles
    axL.set_ylabel("forcing amplitude", fontsize=8)
    axL.tick_params(labelsize=7)
    axL.set_title(TITLE, fontsize=9, fontweight="bold", pad=3)
    pad = 0.06*(amp.max()-amp.min()+1e-9)
    axL.set_ylim(amp.min()-pad, amp.max()+pad); axL.set_xlim(cyc.min(), cyc.max())
    for sp in ("top","right"): axL.spines[sp].set_visible(False)

txt = fig.text(LABEL_XY[0], LABEL_XY[1], "", fontsize=12, fontweight="bold", family="monospace")

# ANIMATE
frames, dyn = [], []
for f in range(len(traj_emb)):
    for a in dyn: a.remove()
    dyn = []; c = int(traj_clu[f])

    mb = keep & (labels_np == c)
    dyn.append(ax.scatter(emb_pts[mb,0], emb_pts[mb,1], emb_pts[mb,2], s=BRIGHT_SIZE,
               c=cluster_colors[c], alpha=BRIGHT_ALPHA, edgecolors="none",
               depthshade=False, zorder=6))
    if f >= 1:
        vp = traj_emb[:f+1]
        dyn.append(ax.plot(vp[:,0], vp[:,1], vp[:,2], color=PATH_COLOR, lw=1.0,
                   alpha=PATH_ALPHA, zorder=7)[0])
    tr = traj_emb[max(0,f-TRAIL_LEN):f+1]
    if len(tr) >= 2:
        dyn.append(ax.plot(tr[:,0], tr[:,1], tr[:,2], color=cluster_colors[c],
                   lw=2.6, alpha=0.95, zorder=8)[0])
    hz = traj_emb[f]
    dyn.append(ax.scatter([hz[0]],[hz[1]],[hz[2]], s=HEAD_SIZE, c=cluster_colors[c],
               marker="o", edgecolors="black", linewidths=1.4, depthshade=False, zorder=9))

    if SHOW_LAW:
        dyn.append(axL.scatter(cyc[:f+1], amp[:f+1],
                   c=[cluster_colors[int(k)] for k in traj_clu[:f+1]],
                   s=20, alpha=0.95, edgecolors="none", zorder=3))
        dyn.append(axL.scatter([cyc[f]], [amp[f]], s=60, c=cluster_colors[c],
                   edgecolors="black", linewidths=1.1, zorder=4))

    im_handle.set_data(disp_imgs[f])
    txt.set_text(f"{REGIME}   t = {cyc[f]:5.2f} cycles   C{c}")   # <<< cycles
    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[...,:3].copy())

# final held frame
for a in dyn: a.remove()
ax.plot(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2], color=PATH_COLOR, lw=1.0,
        alpha=PATH_ALPHA, zorder=7)
ax.scatter(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2], s=FINAL_PT_SIZE,
           c=[cluster_colors[int(c)] for c in traj_clu], alpha=0.95,
           edgecolors="none", depthshade=False, zorder=8)
if SHOW_LAW:
    axL.scatter(cyc, amp, c=[cluster_colors[int(c)] for c in traj_clu],
                s=20, alpha=0.95, edgecolors="none", zorder=3)
im_handle.set_data(disp_imgs[-1])
txt.set_text(f"{REGIME}   full trajectory  (t = 0.00 - {CYC_END:.2f} cycles)")   # <<< cycles
fig.canvas.draw()
frames.extend([np.asarray(fig.canvas.buffer_rgba())[...,:3].copy()] * max(1, int(HOLD_SECONDS*FPS)))

# SAVE
imageio.mimsave(GIF_PATH, frames, fps=FPS, loop=1)
print("wrote", GIF_PATH, f"({len(frames)} frames)")
try:
    imageio.mimsave(MP4_PATH, frames, fps=FPS, codec="libx264", quality=8, macro_block_size=16)
    print("wrote", MP4_PATH)
except Exception as e:
    print("mp4 skipped -- pip install imageio[ffmpeg]  (", e, ")")
plt.close(fig)

[hold] 400 frames  (forcing rows: 20000)
wrote ./gifs/hold_latent_evolution.gif (433 frames)
mp4 skipped -- pip install imageio[ffmpeg]  ( Could not find a backend to open `./gifs/hold_latent_evolution.mp4`` with iomode `wI`.
Based on the extension, the following plugins might add capable backends:
  FFMPEG:  pip install imageio[ffmpeg]
  pyav:  pip install imageio[pyav] )


In [6]:
# Latent-space evolution GIF -- baseline / hold / escape. Local paths, cycles.
# ASSUMES already defined (from the setup / earlier cells): os, np, torch, cv2,
# plt, pe, proj3d, imageio, load_file, model, latents_d8_k9, centroids_d8_k9,
# labels_d8_k9, reducer_3d, cluster_colors, and the helpers encode_and_field,
# clean_field, screen_xy, read_forcing_txt. Nothing re-imported or re-defined.

REGIME = "escape"          # "baseline" | "hold" | "escape"

CFG = {
  "baseline": dict(run_dir=None,
                   snap_dir="./data/baseline",
                   snap_tmpl="snapshot_{it}.png",          # unpadded (baseline)
                   it=(80_000, 100_000, 50), title="baseline"),
  "hold":     dict(run_dir="./data/top_c0_runs/classic-sweep-4",
                   snap_dir="./data/top_c0_runs/classic-sweep-4/snapshots",
                   snap_tmpl="snapshot_{it:06d}.png",      # padded (confirm)
                   it=None, title="hold  (lock-on)"),
  "escape":   dict(run_dir="./data/top_c6_runs/worthy-sweep-43",
                   snap_dir="./data/top_c6_runs/worthy-sweep-43/snapshots",
                   snap_tmpl="snapshot_{it:06d}.png",      # padded (confirm)
                   it=None, title="escape  (relaxation oscillator)"),
}
cfg      = CFG[REGIME]
SNAP_DIR = cfg["snap_dir"]; TITLE = cfg["title"]; RUN_DIR = cfg["run_dir"]
SNAP_TMPL = cfg["snap_tmpl"]
SHOW_LAW = RUN_DIR is not None
FORCING_TXT = os.path.join(RUN_DIR, "rbf_forcing.txt") if SHOW_LAW else None

SAVE_DIR = "./gifs"; os.makedirs(SAVE_DIR, exist_ok=True)
GIF_PATH = os.path.join(SAVE_DIR, f"{REGIME}_latent_evolution.gif")
MP4_PATH = os.path.join(SAVE_DIR, f"{REGIME}_latent_evolution.mp4")

# knobs
MARGIN_THRESHOLD = 2.0
VIEW_ELEV, VIEW_AZIM = 0, 50
ZOOM = 0.88
SOLID_MAX, DASH_MAX = 1.75, 1.85
DT = 0.001
T_SH = 1.667; ITER_PER_CYCLE = T_SH / DT               # shedding-cycle conversion
FRAME_STRIDE = 2; FPS = 11; TRAIL_LEN = 18
HOLD_SECONDS, FINAL_PT_SIZE = 3.0, 20

FIGSIZE = (10.6, 7.0) if SHOW_LAW else (7.6, 7.0)
AX_RECT  = [-0.03, -0.05, 0.72, 1.10] if SHOW_LAW else [-0.06, -0.06, 1.12, 1.12]
TAG_RECT = [0.71, 0.70, 0.27, 0.16]  if SHOW_LAW else [0.655, 0.775, 0.335, 0.185]
LAW_RECT = [0.735, 0.12, 0.245, 0.34]
LABEL_XY = (0.71, 0.90) if SHOW_LAW else (0.03, 0.95)
BG_ALPHA, BG_SIZE = 0.14, 5
BRIGHT_ALPHA, BRIGHT_SIZE = 0.95, 9
CEN_SIZE, HEAD_SIZE = 46, 95
EDGE_COLOR, EDGE_ALPHA = "0.72", 0.9
PATH_COLOR, PATH_ALPHA = "0.55", 0.5
LAW_DIM_ALPHA, LAW_LINE_COLOR = 0.15, "0.8"

# precomputed embeddings (arrays/reducer already in memory)
latents_np   = latents_d8_k9.numpy().astype(np.float32)
centroids_np = centroids_d8_k9.numpy().astype(np.float32)
labels_np    = labels_d8_k9.numpy().astype(int)
DEVICE = next(model.parameters()).device
emb_pts = reducer_3d.transform(latents_np)
emb_cen = reducer_3d.transform(centroids_np)
Dc = np.linalg.norm(centroids_np[:,None,:] - centroids_np[None,:,:], axis=2)
D  = np.linalg.norm(latents_np[:,None,:] - centroids_np[None,:,:], axis=2)
own = D[np.arange(len(labels_np)), labels_np]
D2 = D.copy(); D2[np.arange(len(labels_np)), labels_np] = np.inf
keep = np.ones(len(labels_np), bool) if MARGIN_THRESHOLD is None else ((D2.min(1)/own) >= MARGIN_THRESHOLD)

# TIMELINE
if SHOW_LAW:
    its_full, amp_full = read_forcing_txt(FORCING_TXT)
else:
    a,b,s = cfg["it"]; its_full = np.arange(a, b, s); amp_full = None
sel = np.arange(len(its_full))[::FRAME_STRIDE]
its = its_full[sel]; amp = amp_full[sel] if SHOW_LAW else None

paths, keep_i = [], []
for n, it in enumerate(its):
    p = os.path.join(SNAP_DIR, SNAP_TMPL.format(it=int(it)))
    if os.path.exists(p): paths.append(p); keep_i.append(n)
keep_i = np.array(keep_i, int)
its = its[keep_i]; amp = amp[keep_i] if SHOW_LAW else None
assert len(paths) > 0, f"No snapshots in {SNAP_DIR} matching {SNAP_TMPL}"
print(f"[{REGIME}] {len(paths)} frames")

traj_z, disp_imgs = [], []
for p in paths:
    z, arr = encode_and_field(p)
    traj_z.append(z); disp_imgs.append(clean_field(arr))
traj_z = np.asarray(traj_z, np.float32)
traj_emb = reducer_3d.transform(traj_z)
traj_clu = np.argmin(np.sum((traj_z[:,None,:] - centroids_np[None,:,:])**2, axis=2), axis=1)

it0 = int(its[0]); cyc = (its.astype(float) - it0) / ITER_PER_CYCLE   # cycles from 0
CYC_END = cyc[-1]

# STATIC BACKGROUND
fig = plt.figure(figsize=FIGSIZE, dpi=110, facecolor="white")
ax  = fig.add_axes([0,0,1,1], projection="3d", facecolor="white"); ax.set_axis_off()
halo = [pe.withStroke(linewidth=2.5, foreground="white")]
for k in range(9):
    m = keep & (labels_np == k)
    ax.scatter(emb_pts[m,0], emb_pts[m,1], emb_pts[m,2], s=BG_SIZE,
               c=cluster_colors[k], alpha=BG_ALPHA, edgecolors="none", depthshade=False)
ax.scatter(emb_cen[:,0], emb_cen[:,1], emb_cen[:,2], s=CEN_SIZE,
           c=[cluster_colors[k] for k in range(9)], marker="D",
           edgecolors="black", linewidths=1.0, depthshade=False, zorder=5)
ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
try: ax.set_box_aspect((np.ptp(emb_pts[:,0]), np.ptp(emb_pts[:,1]), np.ptp(emb_pts[:,2])))
except Exception: pass
for setlim, lo, hi in [
    (ax.set_xlim3d, emb_pts[:,0].min(), emb_pts[:,0].max()),
    (ax.set_ylim3d, emb_pts[:,1].min(), emb_pts[:,1].max()),
    (ax.set_zlim3d, emb_pts[:,2].min(), emb_pts[:,2].max())]:
    c = 0.5*(lo+hi); r = 0.5*(hi-lo)*ZOOM; setlim(c-r, c+r)
EDGES = [(5,2),(2,8),(8,1),(1,3),(3,5),(2,3),(5,8),
         (0,1),(0,2),(0,3),(0,5),(0,8),(0,4),(6,4),(6,7)]
EDGES = [(i,j) for (i,j) in EDGES if Dc[i,j] <= DASH_MAX]
_w = np.array([Dc[i,j] for i,j in EDGES]); wmin, wmax = _w.min(), _w.max()
def _lw(d): return 2.4 + (0.0 if wmax==wmin else (d-wmin)/(wmax-wmin))*(0.6-2.4)
for i,j in EDGES:
    style = "-" if Dc[i,j] < SOLID_MAX else "--"; xi, xj = emb_cen[i], emb_cen[j]
    ax.plot([xi[0],xj[0]],[xi[1],xj[1]],[xi[2],xj[2]], color=EDGE_COLOR,
            lw=_lw(Dc[i,j]), linestyle=style, alpha=EDGE_ALPHA, zorder=3)
ax.set_position(AX_RECT); fig.canvas.draw()
for k in range(9):
    s = screen_xy(ax, emb_cen[k])
    ax.annotate(f"C{k}", xy=(s[0],s[1]), xytext=(0,9), textcoords="offset points",
                color=cluster_colors[k], fontsize=11, fontweight="bold",
                zorder=30, annotation_clip=False, path_effects=halo)

axim = fig.add_axes(TAG_RECT); axim.set_xticks([]); axim.set_yticks([])
for sp in axim.spines.values(): sp.set_edgecolor("0.7"); sp.set_linewidth(1.0)
im_handle = axim.imshow(disp_imgs[0], aspect="auto")

if SHOW_LAW:
    axL = fig.add_axes(LAW_RECT)
    axL.plot(cyc, amp, color=LAW_LINE_COLOR, lw=0.9, alpha=0.7, zorder=1)
    axL.scatter(cyc, amp, c=[cluster_colors[int(c)] for c in traj_clu],
                s=16, alpha=LAW_DIM_ALPHA, edgecolors="none", zorder=2)
    axL.set_xlabel("time  (shedding cycles)", fontsize=8)
    axL.set_ylabel("forcing amplitude", fontsize=8)
    axL.tick_params(labelsize=7)
    axL.set_title(TITLE, fontsize=9, fontweight="bold", pad=3)
    pad = 0.06*(amp.max()-amp.min()+1e-9)
    axL.set_ylim(amp.min()-pad, amp.max()+pad); axL.set_xlim(cyc.min(), cyc.max())
    for sp in ("top","right"): axL.spines[sp].set_visible(False)

txt = fig.text(LABEL_XY[0], LABEL_XY[1], "", fontsize=12, fontweight="bold", family="monospace")

# ANIMATE
frames, dyn = [], []
for f in range(len(traj_emb)):
    for a in dyn: a.remove()
    dyn = []; c = int(traj_clu[f])
    mb = keep & (labels_np == c)
    dyn.append(ax.scatter(emb_pts[mb,0], emb_pts[mb,1], emb_pts[mb,2], s=BRIGHT_SIZE,
               c=cluster_colors[c], alpha=BRIGHT_ALPHA, edgecolors="none",
               depthshade=False, zorder=6))
    if f >= 1:
        vp = traj_emb[:f+1]
        dyn.append(ax.plot(vp[:,0], vp[:,1], vp[:,2], color=PATH_COLOR, lw=1.0,
                   alpha=PATH_ALPHA, zorder=7)[0])
    tr = traj_emb[max(0,f-TRAIL_LEN):f+1]
    if len(tr) >= 2:
        dyn.append(ax.plot(tr[:,0], tr[:,1], tr[:,2], color=cluster_colors[c],
                   lw=2.6, alpha=0.95, zorder=8)[0])
    hz = traj_emb[f]
    dyn.append(ax.scatter([hz[0]],[hz[1]],[hz[2]], s=HEAD_SIZE, c=cluster_colors[c],
               marker="o", edgecolors="black", linewidths=1.4, depthshade=False, zorder=9))
    if SHOW_LAW:
        dyn.append(axL.scatter(cyc[:f+1], amp[:f+1],
                   c=[cluster_colors[int(k)] for k in traj_clu[:f+1]],
                   s=20, alpha=0.95, edgecolors="none", zorder=3))
        dyn.append(axL.scatter([cyc[f]], [amp[f]], s=60, c=cluster_colors[c],
                   edgecolors="black", linewidths=1.1, zorder=4))
    im_handle.set_data(disp_imgs[f])
    txt.set_text(f"{REGIME}   t = {cyc[f]:5.2f} cycles   C{c}")
    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[...,:3].copy())

# final held frame
for a in dyn: a.remove()
ax.plot(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2], color=PATH_COLOR, lw=1.0,
        alpha=PATH_ALPHA, zorder=7)
ax.scatter(traj_emb[:,0], traj_emb[:,1], traj_emb[:,2], s=FINAL_PT_SIZE,
           c=[cluster_colors[int(c)] for c in traj_clu], alpha=0.95,
           edgecolors="none", depthshade=False, zorder=8)
if SHOW_LAW:
    axL.scatter(cyc, amp, c=[cluster_colors[int(c)] for c in traj_clu],
                s=20, alpha=0.95, edgecolors="none", zorder=3)
im_handle.set_data(disp_imgs[-1])
txt.set_text(f"{REGIME}   full trajectory  (t = 0.00 - {CYC_END:.2f} cycles)")
fig.canvas.draw()
frames.extend([np.asarray(fig.canvas.buffer_rgba())[...,:3].copy()] * max(1, int(HOLD_SECONDS*FPS)))

# SAVE
imageio.mimsave(GIF_PATH, frames, fps=FPS, loop=1)
print("wrote", GIF_PATH, f"({len(frames)} frames)")
try:
    imageio.mimsave(MP4_PATH, frames, fps=FPS, codec="libx264", quality=8, macro_block_size=16)
    print("wrote", MP4_PATH)
except Exception as e:
    print("mp4 skipped -- pip install imageio[ffmpeg]  (", e, ")")
plt.close(fig)

[escape] 400 frames
wrote ./gifs/escape_latent_evolution.gif (433 frames)
mp4 skipped -- pip install imageio[ffmpeg]  ( Could not find a backend to open `./gifs/escape_latent_evolution.mp4`` with iomode `wI`.
Based on the extension, the following plugins might add capable backends:
  FFMPEG:  pip install imageio[ffmpeg]
  pyav:  pip install imageio[pyav] )


# GitHub visualization

In [7]:
# COMBINED LATENT-EVOLUTION GIF  --  2 rows x 3 columns, one shared clock
#
#   row 1 :  baseline (unforced)  |  C0 hold (lock-on)  |  C6 escape (relaxation osc.)
#            3-D UMAP skeleton + trajectory, vorticity inset on top of each panel
#   row 2 :  (empty)              |  C0 forcing law     |  C6 forcing law
#            law coloured by cluster assignment + marker riding the law
#
#   Every panel shows the SAME CFD iteration in every frame, so the three
#   trajectories and the two laws advance together.
#   Time is in baseline shedding cycles, t/T_sh, counted from the window start
#   (T_sh = 1/f_shed = 1/0.6 = 1.667 t.u. = 1667 iterations at dt = 0.001;
#    the 80k-100k window = 20 t.u. = 12 shedding cycles).
#
# Needs from the setup cell: model, latents_d8_k9, centroids_d8_k9, labels_d8_k9,
#   reducer_3d, cluster_colors, and os, np, torch, cv2, plt, pe, proj3d, Image, imageio.

import re
from functions import load_file          # SAME loader/crop as the Markov analysis

# REGIMES (column order)
REGIMES = [
    dict(key="baseline", title="baseline  (unforced)",
         run_dir=None,                              snap_dir="./data/baseline"),
    dict(key="hold",     title=r"$C_0$ hold  (lock-on)",
         run_dir="./data/top_c0_runs/classic-sweep-4",   snap_dir="./data/top_c0_runs/classic-sweep-4/snapshots"),
    dict(key="escape",   title=r"$C_6$ escape  (relaxation oscillator)",
         run_dir="./data/top_c6_runs/worthy-sweep-43",   snap_dir="./data/top_c6_runs/worthy-sweep-43/snapshots"),
]

SAVE_DIR = "./gifs"; os.makedirs(SAVE_DIR, exist_ok=True)
GIF_PATH = os.path.join(SAVE_DIR, "combined_latent_evolution.gif")
MP4_PATH = os.path.join(SAVE_DIR, "combined_latent_evolution.mp4")

# TIMELINE
IT_START, IT_END, IT_STEP = 80_000, 100_000, 50    # snapshot window (same as analysis)
FRAME_STRIDE = 2                                   # every 2nd snapshot -> 200 frames
DT = 0.001
try:                                               # single source of truth if present
    T_SH = float(np.load("timescales.npy", allow_pickle=True).item()["T_shed"])
except Exception:
    T_SH = 1.0 / 0.6                               # f_shed = 0.6 from the baseline C_L FFT
ITER_PER_CYCLE = T_SH / DT

# UMAP GEOMETRY (identical to the single-regime GIF cells)
MARGIN_THRESHOLD = 2.0
VIEW_ELEV, VIEW_AZIM = 0, 50
ZOOM = 0.88
SOLID_MAX, DASH_MAX = 1.75, 1.85
EDGES = [(5,2),(2,8),(8,1),(1,3),(3,5),(2,3),(5,8),
         (0,1),(0,2),(0,3),(0,5),(0,8),(0,4),(6,4),(6,7)]

# PACING
FPS, TRAIL_LEN, HOLD_SECONDS = 11, 18, 3.0
LOOP = 1                                           # same as your GIFs (0 = loop forever)

# STYLING (identical to the single-regime GIF cells)
BG_ALPHA, BG_SIZE = 0.14, 5
BRIGHT_ALPHA, BRIGHT_SIZE = 0.95, 9
CEN_SIZE, HEAD_SIZE, FINAL_PT_SIZE = 46, 95, 20
EDGE_COLOR, EDGE_ALPHA = "0.72", 0.9
PATH_COLOR, PATH_ALPHA = "0.55", 0.5
LAW_DIM_ALPHA, LAW_LINE_COLOR = 0.15, "0.8"
SHARE_LAW_Y = True                                 # C0 and C6 laws on the same b-axis

# LAYOUT (figure fractions)
FIGSIZE, DPI = (18.0, 10.5), 90
COL_X0, COL_W = [0.0, 1/3, 2/3], 1/3
UMAP_RECT = lambda x0: [x0 - 0.012, 0.215, COL_W + 0.024, 0.655]   # row 1, 3-D axes
TAG_RECT  = lambda x0: [x0 + 0.205, 0.785, 0.118, 0.105]           # vorticity inset
LAW_RECT  = lambda x0: [x0 + 0.055, 0.075, 0.255, 0.175]           # row 2, law axes
TITLE_Y, STATE_Y, CLOCK_Y = 0.935, 0.895, 0.975


# HELPERS
def snapshot_index(folder):
    """{iteration: path}; accepts folder or folder/snapshots, padded or unpadded names."""
    for d in (os.path.join(folder, "snapshots"), folder):
        if not os.path.isdir(d):
            continue
        idx = {}
        for f in os.listdir(d):
            m = re.fullmatch(r"snapshot_(\d+)\.png", f)
            if m:
                idx[int(m.group(1))] = os.path.join(d, f)
        if idx:
            return idx
    raise FileNotFoundError(f"no snapshot_*.png in {folder} or {folder}/snapshots")

def read_forcing(run_dir):
    it, b = [], []
    with open(os.path.join(run_dir, "rbf_forcing.txt")) as f:
        next(f)
        for line in f:
            p = line.strip().split(",")
            if len(p) >= 2:
                it.append(int(p[0])); b.append(float(p[1]))
    it, b = np.array(it), np.array(b)
    w = (it >= IT_START) & (it < IT_END)
    return it[w], b[w]

def clean_field(rgb_u8):
    rgb = rgb_u8.astype(np.float32)
    R, G, B = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    mx = rgb.max(2); mn = rgb.min(2)
    luma = 0.299*R + 0.587*G + 0.114*B
    sat  = (mx - mn) / (mx + 1e-6)
    gray_halo = (sat < 0.16) & (luma > 110)
    mask = ((sat > 0.14) | (luma < 80)); mask[gray_halo] = False
    mask = mask.astype(np.uint8); k = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    alpha = np.clip(cv2.GaussianBlur(mask.astype(np.float32), (0, 0), 0.7), 0, 1)
    rgb[gray_halo] = 255
    return np.clip(rgb*alpha[..., None] + 255*(1 - alpha[..., None]), 0, 255).astype(np.uint8)

def screen_xy(axis, p3):
    x, y, _ = proj3d.proj_transform(p3[0], p3[1], p3[2], axis.get_proj())
    return np.array([x, y])

def to_cyc(it):
    return (np.asarray(it, float) - IT_START) / ITER_PER_CYCLE


# SHARED ARRAYS (background embedding computed once)
latents_np   = latents_d8_k9.numpy().astype(np.float32)
centroids_np = centroids_d8_k9.numpy().astype(np.float32)
labels_np    = labels_d8_k9.numpy().astype(int)
DEVICE = next(model.parameters()).device

emb_pts = reducer_3d.transform(latents_np)
emb_cen = reducer_3d.transform(centroids_np)
Dc  = np.linalg.norm(centroids_np[:, None] - centroids_np[None], axis=2)
D   = np.linalg.norm(latents_np[:, None] - centroids_np[None], axis=2)
own = D[np.arange(len(labels_np)), labels_np]
D2  = D.copy(); D2[np.arange(len(labels_np)), labels_np] = np.inf
keep = np.ones(len(labels_np), bool) if MARGIN_THRESHOLD is None else (D2.min(1) / own >= MARGIN_THRESHOLD)
EDGES_KEPT = [(i, j) for (i, j) in EDGES if Dc[i, j] <= DASH_MAX]
_w = np.array([Dc[i, j] for i, j in EDGES_KEPT] or [1.0]); wmin, wmax = _w.min(), _w.max()
edge_lw = lambda d: 2.4 + (0.0 if wmax == wmin else (d - wmin)/(wmax - wmin))*(0.6 - 2.4)


# COMMON FRAME TIMELINE (only iterations present in ALL three runs)
for R in REGIMES:
    R["snaps"] = snapshot_index(R["snap_dir"])
grid = np.arange(IT_START, IT_END, IT_STEP)
common = [int(it) for it in grid if all(it in R["snaps"] for R in REGIMES)]
for R in REGIMES:
    miss = sum(it not in R["snaps"] for it in grid)
    print(f"[{R['key']:<8}] {len(R['snaps'])} snapshots on disk, {miss} missing in window")
assert common, "no iteration is present in all three runs"
frame_its = np.array(common[::FRAME_STRIDE])
cyc = to_cyc(frame_its)
NF = len(frame_its)
print(f"{NF} frames, t/T_sh = {cyc[0]:.2f} .. {cyc[-1]:.2f}   (T_sh = {T_SH:.4f} t.u.)")


# ENCODE EVERY REGIME ON THE COMMON TIMELINE
model.eval()
for R in REGIMES:
    zs, imgs = [], []
    with torch.no_grad():
        for it in frame_its:
            arr = load_file(R["snaps"][int(it)])
            x = (torch.tensor(arr, dtype=torch.float32)/255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            zs.append(model.encode(x).squeeze(0).cpu().numpy())
            imgs.append(clean_field(np.asarray(arr, dtype=np.uint8)))
    R["z"]    = np.asarray(zs, np.float32)
    R["emb"]  = reducer_3d.transform(R["z"])
    R["clu"]  = np.argmin(np.sum((R["z"][:, None] - centroids_np[None])**2, axis=2), axis=1)
    R["cols"] = np.array([cluster_colors[int(k)] for k in R["clu"]])
    R["imgs"] = imgs
    if R["run_dir"] is not None:
        R["law_it"], R["law_b"] = read_forcing(R["run_dir"])
        R["b"] = np.interp(frame_its, R["law_it"], R["law_b"])   # b(t) at each frame
    print(f"[{R['key']:<8}] encoded {len(zs)} frames; clusters visited:",
          sorted(set(R["clu"].tolist())))

law_regs = [R for R in REGIMES if R["run_dir"] is not None]
if SHARE_LAW_Y:
    b_lo = min(R["law_b"].min() for R in law_regs); b_hi = max(R["law_b"].max() for R in law_regs)
    pad = 0.06*(b_hi - b_lo + 1e-9); SHARED_YLIM = (b_lo - pad, b_hi + pad)


# STATIC FIGURE
fig  = plt.figure(figsize=FIGSIZE, dpi=DPI, facecolor="white")
halo = [pe.withStroke(linewidth=2.5, foreground="white")]
clock = fig.text(0.5, CLOCK_Y, "", ha="center", va="center", fontsize=15,
                 fontweight="bold", family="monospace")

for col, R in enumerate(REGIMES):
    x0 = COL_X0[col]

    # row 1: 3-D UMAP skeleton (same look as the single-regime GIFs)
    ax = fig.add_axes(UMAP_RECT(x0), projection="3d")
    ax.set_facecolor((1, 1, 1, 0)); ax.patch.set_alpha(0); ax.set_axis_off()
    for k in range(9):
        m = keep & (labels_np == k)
        ax.scatter(emb_pts[m, 0], emb_pts[m, 1], emb_pts[m, 2], s=BG_SIZE, c=cluster_colors[k],
                   alpha=BG_ALPHA, edgecolors="none", depthshade=False)
    ax.scatter(emb_cen[:, 0], emb_cen[:, 1], emb_cen[:, 2], s=CEN_SIZE,
               c=[cluster_colors[k] for k in range(9)], marker="D",
               edgecolors="black", linewidths=1.0, depthshade=False, zorder=5)
    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
    try:
        ax.set_box_aspect((np.ptp(emb_pts[:, 0]), np.ptp(emb_pts[:, 1]), np.ptp(emb_pts[:, 2])))
    except Exception:
        pass
    for setlim, dim in [(ax.set_xlim3d, 0), (ax.set_ylim3d, 1), (ax.set_zlim3d, 2)]:
        lo, hi = emb_pts[:, dim].min(), emb_pts[:, dim].max()
        c = 0.5*(lo + hi); r = 0.5*(hi - lo)*ZOOM; setlim(c - r, c + r)
    for i, j in EDGES_KEPT:
        xi, xj = emb_cen[i], emb_cen[j]
        ax.plot([xi[0], xj[0]], [xi[1], xj[1]], [xi[2], xj[2]], color=EDGE_COLOR,
                lw=edge_lw(Dc[i, j]), linestyle="-" if Dc[i, j] < SOLID_MAX else "--",
                alpha=EDGE_ALPHA, zorder=3)

    # per-frame artists, created once and updated in place
    R["bright"] = []
    for k in range(9):
        m = keep & (labels_np == k)
        s = ax.scatter(emb_pts[m, 0], emb_pts[m, 1], emb_pts[m, 2], s=BRIGHT_SIZE, c=cluster_colors[k],
                       alpha=BRIGHT_ALPHA, edgecolors="none", depthshade=False, zorder=6)
        s.set_visible(False); R["bright"].append(s)
    e0 = R["emb"][0]
    R["path"],  = ax.plot([e0[0]], [e0[1]], [e0[2]], color=PATH_COLOR, lw=1.0, alpha=PATH_ALPHA, zorder=7)
    R["trail"], = ax.plot([e0[0]], [e0[1]], [e0[2]], lw=2.6, alpha=0.95, zorder=8)
    R["head"],  = ax.plot([e0[0]], [e0[1]], [e0[2]], linestyle="none", marker="o",
                          markersize=np.sqrt(HEAD_SIZE), markeredgecolor="black",
                          markeredgewidth=1.4, zorder=9)
    R["ax"] = ax

    # vorticity inset (top of the panel)
    axim = fig.add_axes(TAG_RECT(x0)); axim.set_xticks([]); axim.set_yticks([])
    for sp in axim.spines.values(): sp.set_edgecolor("0.7"); sp.set_linewidth(1.0)
    R["im"] = axim.imshow(R["imgs"][0], aspect="auto")

    # panel title + current-cluster label
    fig.text(x0 + 0.015, TITLE_Y, R["title"], fontsize=14, fontweight="bold", va="center")
    R["state"] = fig.text(x0 + 0.015, STATE_Y, "", fontsize=14, fontweight="bold",
                          va="center", family="monospace")

    # row 2: forcing law (hold / escape only; baseline column stays empty)
    if R["run_dir"] is not None:
        axL = fig.add_axes(LAW_RECT(x0))
        axL.plot(to_cyc(R["law_it"]), R["law_b"], color=LAW_LINE_COLOR, lw=0.9, alpha=0.7, zorder=1)
        axL.scatter(cyc, R["b"], c=R["cols"], s=16, alpha=LAW_DIM_ALPHA, edgecolors="none", zorder=2)
        R["law_dots"] = axL.scatter(cyc[:1], R["b"][:1], c=R["cols"][:1], s=20, alpha=0.95,
                                    edgecolors="none", zorder=3)
        R["law_head"], = axL.plot([cyc[0]], [R["b"][0]], linestyle="none", marker="o",
                                  markersize=np.sqrt(60), markeredgecolor="black",
                                  markeredgewidth=1.1, zorder=4)
        axL.set_xlim(0, (IT_END - IT_START) / ITER_PER_CYCLE)      # full window: 0 .. 12 T_sh
        if SHARE_LAW_Y:
            axL.set_ylim(*SHARED_YLIM)
        else:
            p = 0.06*(np.ptp(R["law_b"]) + 1e-9); axL.set_ylim(R["law_b"].min() - p, R["law_b"].max() + p)
        axL.set_xlabel(r"$t\,/\,T_{sh}$", fontsize=12)
        axL.set_ylabel(r"$b(t)$", fontsize=12)
        axL.tick_params(labelsize=10)
        axL.set_title("forcing law", fontsize=12, fontweight="bold", pad=4, loc="left")
        for sp in ("top", "right"): axL.spines[sp].set_visible(False)

# centroid labels need the final axes positions -> draw once, then project
fig.canvas.draw()
for R in REGIMES:
    for k in range(9):
        s = screen_xy(R["ax"], emb_cen[k])
        R["ax"].annotate(f"C{k}", xy=(s[0], s[1]), xytext=(0, 9), textcoords="offset points",
                         color=cluster_colors[k], fontsize=11, fontweight="bold",
                         zorder=30, annotation_clip=False, path_effects=halo)


# ANIMATE (update artists in place; frames stored palettised to save memory)
def grab():
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

try:
    mp4 = imageio.get_writer(MP4_PATH, fps=FPS, codec="libx264", quality=8, macro_block_size=16)
except Exception:
    mp4 = None
    print("mp4 skipped -- pip install imageio[ffmpeg]")

NO_DITHER = getattr(Image, "Dither", Image).NONE      # works on old and new Pillow
gif_frames = []
def store(rgb):
    gif_frames.append(Image.fromarray(rgb).quantize(colors=256, dither=NO_DITHER))
    if mp4 is not None:
        mp4.append_data(rgb)

for f in range(NF):
    for R in REGIMES:
        c = int(R["clu"][f]); col = cluster_colors[c]
        for k in range(9):
            R["bright"][k].set_visible(k == c)
        e = R["emb"]
        R["path"].set_data_3d(e[:f+1, 0], e[:f+1, 1], e[:f+1, 2])
        lo = max(0, f - TRAIL_LEN)
        R["trail"].set_data_3d(e[lo:f+1, 0], e[lo:f+1, 1], e[lo:f+1, 2]); R["trail"].set_color(col)
        R["head"].set_data_3d([e[f, 0]], [e[f, 1]], [e[f, 2]]); R["head"].set_markerfacecolor(col)
        R["im"].set_data(R["imgs"][f])
        R["state"].set_text(f"C{c}"); R["state"].set_color(col)
        if R["run_dir"] is not None:
            R["law_dots"].set_offsets(np.c_[cyc[:f+1], R["b"][:f+1]])
            R["law_dots"].set_facecolor(R["cols"][:f+1])
            R["law_head"].set_data([cyc[f]], [R["b"][f]]); R["law_head"].set_markerfacecolor(col)
    clock.set_text(f"t / T_sh = {cyc[f]:5.2f}")
    store(grab())

# final held frame: whole trajectory coloured by cluster, whole law revealed
for R in REGIMES:
    for s in R["bright"]: s.set_visible(False)
    R["trail"].set_visible(False); R["head"].set_visible(False)
    e = R["emb"]
    R["path"].set_data_3d(e[:, 0], e[:, 1], e[:, 2])
    R["ax"].scatter(e[:, 0], e[:, 1], e[:, 2], s=FINAL_PT_SIZE, c=R["cols"], alpha=0.95,
                    edgecolors="none", depthshade=False, zorder=8)
    R["state"].set_text("")
    if R["run_dir"] is not None:
        R["law_dots"].set_offsets(np.c_[cyc, R["b"]]); R["law_dots"].set_facecolor(R["cols"])
        R["law_head"].set_visible(False)
clock.set_text(f"full trajectories   t / T_sh = {cyc[0]:.2f} - {cyc[-1]:.2f}")
final = grab()
store(final)
if mp4 is not None:
    for _ in range(max(0, int(HOLD_SECONDS*FPS) - 1)):
        mp4.append_data(final)
    mp4.close(); print("wrote", MP4_PATH)

# SAVE GIF (last frame held for HOLD_SECONDS)
durations = [int(round(1000/FPS))]*(len(gif_frames) - 1) + [int(HOLD_SECONDS*1000)]
gif_frames[0].save(GIF_PATH, save_all=True, append_images=gif_frames[1:],
                   duration=durations, loop=LOOP, optimize=False)
plt.close(fig)
print("wrote", GIF_PATH, f"({len(gif_frames)} frames, {os.path.getsize(GIF_PATH)/1e6:.1f} MB)")

[baseline] 400 snapshots on disk, 0 missing in window
[hold    ] 400 snapshots on disk, 0 missing in window
[escape  ] 400 snapshots on disk, 0 missing in window
200 frames, t/T_sh = 0.00 .. 11.94   (T_sh = 1.6667 t.u.)
[baseline] encoded 200 frames; clusters visited: [1, 2, 3, 5, 8]
[hold    ] encoded 200 frames; clusters visited: [0, 1, 2, 3, 5, 8]
[escape  ] encoded 200 frames; clusters visited: [0, 1, 2, 3, 4, 5, 6, 7, 8]
mp4 skipped -- pip install imageio[ffmpeg]
wrote ./gifs/combined_latent_evolution.gif (201 frames, 24.2 MB)
